In [1]:
# ! pip install skl2onnx onnx onnxruntime

In [2]:
# ============================================================
# PHASE 3 - KNOWLEDGE DISTILLATION + ONNX DEPLOYMENT
# ML-Based Predictive Irrigation System for Paddy Cultivation
# Cell 0.1 - Library Imports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import time
import os
import json
import psutil
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import sklearn.base

# Hyperparameter optimisation
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Model persistence
import joblib

# ONNX ecosystem
import onnx
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from onnxruntime.quantization import quantize_dynamic, QuantType

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Global constants
RANDOM_STATE  = 42
ALPHA_DISTILL = 0.8    # teacher soft label weight
N_TRIALS_STU  = 60     # Optuna trials for student MLP

print("=" * 60)
print("PHASE 3 - KNOWLEDGE DISTILLATION + ONNX DEPLOYMENT")
print("ML-Based Predictive Irrigation System for Paddy Cultivation")
print("=" * 60)
print("\nAll libraries imported successfully.")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [3]:
# ============================================================
# Cell 0.2 - Load Phase 2 Hybrid Stack Components
# ============================================================

models_dir    = './Models'
retrained_dir = './Models/retrained'

retrained_family1 = joblib.load(f"{retrained_dir}/retrained_family1_catboost.pkl")
retrained_family2 = joblib.load(f"{retrained_dir}/retrained_family2_svr.pkl")
retrained_family3 = joblib.load(f"{retrained_dir}/retrained_family3_mlp.pkl")
meta_learner      = joblib.load(f"{models_dir}/meta_learner_ridge.pkl")

retrained_champions = [
    (retrained_family1, 'CatBoost'),
    (retrained_family2, 'SVR'),
    (retrained_family3, 'MLP')
]

print("=" * 60)
print("PHASE 2 MODELS LOADED")
print("=" * 60)
print(f"Family 1 : CatBoost  --> {type(retrained_family1).__name__}")
print(f"Family 2 : SVR       --> Pipeline({type(retrained_family2.named_steps['svr']).__name__})")
print(f"Family 3 : MLP       --> Pipeline({type(retrained_family3.named_steps['mlp']).__name__})")
print(f"Meta     : Ridge     --> {type(meta_learner).__name__}")
print("\nAll Phase 2 components loaded successfully.")


In [7]:
# ============================================================
# Cell 0.3 - Reload Dataset and Reproduce Exact Phase 2 Splits
# Same random seed and stratification as Phase 2
# ============================================================

dataset_filename = './Data/paddy_CWR_dataset_20260227_131300.csv'

df = pd.read_csv(dataset_filename)

feature_columns = [
    'Water_Depth_cm',
    'Soil_Moisture_%',
    'Tank_Level_%',
    'ET_mm_day',
    'Rainfall_Predicted_mm',
    'Crop_Stage'
]
target_column = 'CWR_mm'

X = df[feature_columns].copy()
y = df[target_column].copy()

# Reproduce exact Phase 2 splits
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=df['Crop_Stage']
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=RANDOM_STATE,
    stratify=X_temp['Crop_Stage']
)

print("=" * 60)
print("DATASET LOADED AND SPLITS REPRODUCED")
print("=" * 60)
print(f"Dataset       : {dataset_filename}")
print(f"Total samples : {len(df)}")
print(f"Features      : {feature_columns}")
print(f"Target        : {target_column}")
print("-" * 60)
print(f"Train set     : {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val set       : {len(X_val)}  samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test set      : {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")
print("-" * 60)

# Helper function for hybrid stack inference
# Used throughout Phase 3
def teacher_predict(X_input):
    if not isinstance(X_input, pd.DataFrame):
        X_input = pd.DataFrame(X_input, columns=feature_columns)
    meta_features = np.zeros((len(X_input), 3))
    for idx, (champion, _) in enumerate(retrained_champions):
        meta_features[:, idx] = champion.predict(X_input)
    predictions = meta_learner.predict(meta_features)
    return np.clip(predictions, 0, None)    # CWR cannot be negative

print("Teacher prediction function defined: teacher_predict()")
print("\nStep 0 complete. Ready for Step 1 -- Soft Label Generation.")



# Step 1 - Soft label generation

In [8]:
# ============================================================
# STEP 1 - SOFT LABEL GENERATION
# Cell 1.1 - Teacher Predictions on Train + Validation Set
# Teacher = Phase 2 Hybrid Stacking Ensemble
# Generates soft labels on 3,400 samples (train + val)
# Test set remains completely untouched
# ============================================================

print("=" * 60)
print("STEP 1 - SOFT LABEL GENERATION")
print("=" * 60)
print("Generating teacher predictions on Train + Validation set...")
print(f"Train samples : {len(X_train)}")
print(f"Val samples   : {len(X_val)}")
print(f"Total         : {len(X_train) + len(X_val)} samples")
print("-" * 60)

# Combine train and validation sets
X_distill = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_distill = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# Teacher predictions (soft labels)
start_time = time.time()
y_teacher  = teacher_predict(X_distill)
teacher_time = time.time() - start_time

teacher_mae_distill = mean_absolute_error(y_distill, y_teacher)

print(f"Teacher predictions generated in {teacher_time:.2f}s")
print(f"\nTeacher Soft Label Statistics:")
print(f"  Min    : {y_teacher.min():.4f} mm")
print(f"  Max    : {y_teacher.max():.4f} mm")
print(f"  Mean   : {y_teacher.mean():.4f} mm")
print(f"  Std    : {y_teacher.std():.4f} mm")
print(f"\nTeacher MAE on distillation set : {teacher_mae_distill:.4f} mm")
print("(Expected to be low -- teacher has seen train set before)")


In [9]:
# ============================================================
# Cell 1.2 - Blend Soft Labels with Ground Truth
# Blended target = 0.8 * teacher prediction
#                + 0.2 * original CWR ground truth
# Prevents student from inheriting teacher errors
# ============================================================

print("=" * 60)
print("SOFT LABEL BLENDING")
print("=" * 60)
print(f"Alpha (teacher weight)       : {ALPHA_DISTILL}")
print(f"1 - Alpha (ground truth weight): {1 - ALPHA_DISTILL}")
print(f"Formula: target = {ALPHA_DISTILL} * teacher + {1 - ALPHA_DISTILL} * ground_truth")
print("-" * 60)

# Blended distillation targets
y_distill_blended = (
    ALPHA_DISTILL * y_teacher +
    (1 - ALPHA_DISTILL) * y_distill.values
)
# Clip to physical lower bound -- CWR cannot be negative
y_distill_blended = np.clip(y_distill_blended, 0, None)

# Comparison statistics
print(f"\nDistillation Target Statistics:")
print(f"{'Metric':<20} {'Ground Truth':>14} {'Teacher':>14} {'Blended':>14}")
print("-" * 65)
print(f"{'Min':<20} {y_distill.min():>14.4f} {y_teacher.min():>14.4f} {y_distill_blended.min():>14.4f}")
print(f"{'Max':<20} {y_distill.max():>14.4f} {y_teacher.max():>14.4f} {y_distill_blended.max():>14.4f}")
print(f"{'Mean':<20} {y_distill.mean():>14.4f} {y_teacher.mean():>14.4f} {y_distill_blended.mean():>14.4f}")
print(f"{'Std':<20} {y_distill.std():>14.4f} {y_teacher.std():>14.4f} {y_distill_blended.std():>14.4f}")

# Visualise distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(y_distill.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Ground Truth CWR')
axes[0].set_xlabel('CWR (mm)')
axes[0].set_ylabel('Frequency')

axes[1].hist(y_teacher, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('Teacher Soft Labels')
axes[1].set_xlabel('CWR (mm)')

axes[2].hist(y_distill_blended, bins=50, edgecolor='black', alpha=0.7)
axes[2].set_title('Blended Distillation Targets')
axes[2].set_xlabel('CWR (mm)')

plt.suptitle(
    f'Distillation Target Distributions (alpha={ALPHA_DISTILL})',
    fontsize=13
)
plt.tight_layout()
plt.savefig('./Visualisation/distillation_targets.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nSaved: distillation_targets.png")
print("\nStep 1 complete.")
print("Blended targets ready in --> y_distill_blended")
print("Proceeding to Step 2 -- Student MLP Training.")


# Step 2 - Student MLP Training with Optuna

In [10]:
# ============================================================
# STEP 2 - STUDENT MLP TRAINING
# Cell 2.1 - Student MLP Architecture Search with Optuna
# Constrained search space for Pi-friendly architecture
# Max 2 hidden layers, max 64 neurons per layer
# Trains on blended distillation targets (y_distill_blended)
# ============================================================

N_TRIALS_STU = 60

def objective_student(trial):

    # Constrained architecture -- Pi friendly
    n_layers = trial.suggest_int('n_layers', 1, 2)
    layers   = []
    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_layer_{i}', 16, 64)
        layers.append(neurons)
    hidden_layer_sizes = tuple(layers)

    params = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation':         trial.suggest_categorical('activation', ['relu', 'tanh']),
        'alpha':              trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True),
        'learning_rate':      trial.suggest_categorical('learning_rate', ['constant', 'adaptive']),
        'max_iter':           500,
        'early_stopping':     True,
        'validation_fraction': 0.1,
        'n_iter_no_change':   20,
        'random_state':       RANDOM_STATE
    }

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp',    MLPRegressor(**params))
    ])

    # Train on blended distillation targets
    pipeline.fit(X_distill, y_distill_blended)

    # Evaluate on validation set using original ground truth
    val_pred = pipeline.predict(X_val)
    val_pred = np.clip(val_pred, 0, None)
    return mean_absolute_error(y_val, val_pred)

print("=" * 60)
print("STEP 2 - STUDENT MLP TRAINING")
print("=" * 60)
print("Search space:")
print("  Hidden layers : 1 to 2")
print("  Neurons/layer : 16 to 64")
print("  Activation    : relu, tanh")
print("  Max iterations: 500")
print(f"  Optuna trials : {N_TRIALS_STU}")
print("-" * 60)
print("Optimizing Student MLP...")

start_time = time.time()
study_student = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_student.optimize(
    objective_student,
    n_trials=N_TRIALS_STU,
    show_progress_bar=True,
    timeout=600
)
student_tune_time = time.time() - start_time

print(f"\nBest Validation MAE : {study_student.best_value:.4f} mm")
print(f"Tuning Time         : {student_tune_time:.1f}s")
print(f"Best Parameters     : {study_student.best_params}")


In [11]:
# ============================================================
# Cell 2.2 - Build and Fit Final Student Model
# Reconstruct best architecture from Optuna params
# Fit on full distillation set (train + val blended targets)
# ============================================================

best_student_params = study_student.best_params

# Reconstruct hidden_layer_sizes from Optuna flat params
n_layers_student = best_student_params['n_layers']
hidden_layer_sizes_student = tuple(
    best_student_params[f'neurons_layer_{i}']
    for i in range(n_layers_student)
)

# Final student MLPRegressor params
student_mlp_params = {
    'hidden_layer_sizes': hidden_layer_sizes_student,
    'activation':         best_student_params['activation'],
    'alpha':              best_student_params['alpha'],
    'learning_rate_init': best_student_params['learning_rate_init'],
    'learning_rate':      best_student_params['learning_rate'],
    'max_iter':           500,
    'early_stopping':     True,
    'validation_fraction': 0.1,
    'n_iter_no_change':   20,
    'random_state':       RANDOM_STATE
}

# Build final student pipeline
student_model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp',    MLPRegressor(**student_mlp_params))
])

# Fit on full distillation set
start_time = time.time()
student_model.fit(X_distill, y_distill_blended)
student_fit_time = time.time() - start_time

# Validation MAE on original ground truth
student_val_pred = np.clip(student_model.predict(X_val), 0, None)
student_val_mae  = mean_absolute_error(y_val, student_val_pred)

print("=" * 60)
print("STUDENT MLP - FINAL MODEL SUMMARY")
print("=" * 60)
print(f"Architecture        : {hidden_layer_sizes_student}")
print(f"Activation          : {best_student_params['activation']}")
print(f"Learning Rate Init  : {best_student_params['learning_rate_init']:.6f}")
print(f"Learning Rate Type  : {best_student_params['learning_rate']}")
print(f"Alpha (L2 reg)      : {best_student_params['alpha']:.6f}")
print(f"Training Time       : {student_fit_time:.2f}s")
print(f"Validation MAE      : {student_val_mae:.4f} mm")
print("-" * 60)

# Parameter count
n_params = sum(
    w.size for w in student_model.named_steps['mlp'].coefs_ +
    student_model.named_steps['mlp'].intercepts_
)
print(f"Total Parameters    : {n_params}")
print("\nStudent model saved as --> student_model")
print("Proceeding to Step 3 -- Student vs Teacher Evaluation.")


# Step 3 - Student vs Teacher Evaluation

In [12]:
# ============================================================
# STEP 3 - STUDENT VS TEACHER EVALUATION
# Cell 3.1 - Test Set Metrics for Student and Teacher
# Both evaluated on the same held-out test set (600 samples)
# Test set was never used in distillation training
# ============================================================

print("=" * 60)
print("STEP 3 - STUDENT VS TEACHER EVALUATION")
print("=" * 60)
print(f"Evaluation set : Test set ({len(X_test)} samples)")
print("-" * 60)

# Teacher predictions on test set
y_pred_teacher = np.clip(teacher_predict(X_test), 0, None)

# Student predictions on test set
y_pred_student = np.clip(student_model.predict(X_test), 0, None)

# Compute metrics
def compute_metrics(y_true, y_pred, model_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {
        'Model': model_name,
        'MAE':   mae,
        'RMSE':  rmse,
        'R2':    r2
    }

teacher_metrics = compute_metrics(y_test, y_pred_teacher, 'Hybrid Stack (Teacher)')
student_metrics = compute_metrics(y_test, y_pred_student, 'Student MLP')

print(f"{'Model':<28} {'MAE (mm)':>10} {'RMSE (mm)':>10} {'R2':>8}")
print("-" * 60)
for m in [teacher_metrics, student_metrics]:
    print(f"{m['Model']:<28} {m['MAE']:>10.4f} {m['RMSE']:>10.4f} {m['R2']:>8.4f}")
print("-" * 60)


In [13]:
# ============================================================
# Cell 3.2 - Distillation Fidelity Report
# Accuracy drop between teacher and student
# Fidelity between teacher and student predictions
# ============================================================

# Accuracy drop metrics
mae_drop  = student_metrics['MAE']  - teacher_metrics['MAE']
rmse_drop = student_metrics['RMSE'] - teacher_metrics['RMSE']
r2_drop   = teacher_metrics['R2']   - student_metrics['R2']

# Fidelity -- how closely student mimics teacher predictions
fidelity_mae  = mean_absolute_error(y_pred_teacher, y_pred_student)
fidelity_corr = np.corrcoef(y_pred_teacher, y_pred_student)[0, 1]

print("=" * 60)
print("DISTILLATION FIDELITY REPORT")
print("=" * 60)
print(f"\nAccuracy Drop (Student - Teacher):")
print(f"  MAE  drop : +{mae_drop:.4f} mm")
print(f"  RMSE drop : +{rmse_drop:.4f} mm")
print(f"  R2   drop : -{r2_drop:.4f}")
print(f"\nStudent-Teacher Fidelity:")
print(f"  MAE between teacher and student predictions : {fidelity_mae:.4f} mm")
print(f"  Correlation of predictions                  : {fidelity_corr:.6f}")
print("-" * 60)

if mae_drop <= 0.5:
    print("\nDistillation quality : Excellent (MAE drop <= 0.5 mm)")
elif mae_drop <= 1.0:
    print("\nDistillation quality : Good (MAE drop <= 1.0 mm)")
elif mae_drop <= 2.0:
    print("\nDistillation quality : Acceptable (MAE drop <= 2.0 mm)")
else:
    print("\nDistillation quality : Poor (MAE drop > 2.0 mm)")
    print("Consider increasing N_TRIALS_STU or relaxing architecture constraints.")


In [14]:
# ============================================================
# Cell 3.3 - Residual and Prediction Comparison Plots
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Plot 1: Predicted vs Actual -- Teacher ---
axes[0, 0].scatter(y_test, y_pred_teacher, alpha=0.4, s=15)
axes[0, 0].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', linewidth=2, label='Perfect prediction'
)
axes[0, 0].set_xlabel('Actual CWR (mm)')
axes[0, 0].set_ylabel('Predicted CWR (mm)')
axes[0, 0].set_title(f'Teacher: Predicted vs Actual\nMAE={teacher_metrics["MAE"]:.4f} mm')
axes[0, 0].legend()

# --- Plot 2: Predicted vs Actual -- Student ---
axes[0, 1].scatter(y_test, y_pred_student, alpha=0.4, s=15)
axes[0, 1].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', linewidth=2, label='Perfect prediction'
)
axes[0, 1].set_xlabel('Actual CWR (mm)')
axes[0, 1].set_ylabel('Predicted CWR (mm)')
axes[0, 1].set_title(f'Student: Predicted vs Actual\nMAE={student_metrics["MAE"]:.4f} mm')
axes[0, 1].legend()

# --- Plot 3: Residuals -- Teacher ---
teacher_residuals = y_test.values - y_pred_teacher
axes[1, 0].scatter(y_pred_teacher, teacher_residuals, alpha=0.4, s=15)
axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Predicted CWR (mm)')
axes[1, 0].set_ylabel('Residual (mm)')
axes[1, 0].set_title('Teacher: Residual Plot')

# --- Plot 4: Residuals -- Student ---
student_residuals = y_test.values - y_pred_student
axes[1, 1].scatter(y_pred_student, student_residuals, alpha=0.4, s=15)
axes[1, 1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Predicted CWR (mm)')
axes[1, 1].set_ylabel('Residual (mm)')
axes[1, 1].set_title('Student: Residual Plot')

plt.suptitle(
    'Teacher vs Student -- Prediction and Residual Analysis',
    fontsize=14
)
plt.tight_layout()
plt.savefig('./Visualisation/student_teacher_evaluation.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: student_teacher_evaluation.png")


In [15]:
# ============================================================
# Cell 3.4 - Teacher vs Student Prediction Agreement
# Shows how closely student mimics teacher on test set
# High correlation = successful knowledge transfer
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Teacher vs Student predictions scatter ---
axes[0].scatter(y_pred_teacher, y_pred_student, alpha=0.4, s=15)
axes[0].plot(
    [y_pred_teacher.min(), y_pred_teacher.max()],
    [y_pred_teacher.min(), y_pred_teacher.max()],
    'r--', linewidth=2, label='Perfect agreement'
)
axes[0].set_xlabel('Teacher Predictions (mm)')
axes[0].set_ylabel('Student Predictions (mm)')
axes[0].set_title(
    f'Teacher vs Student Agreement\n'
    f'Fidelity MAE={fidelity_mae:.4f} mm  |  '
    f'Correlation={fidelity_corr:.4f}'
)
axes[0].legend()

# --- Plot 2: Prediction difference distribution ---
pred_diff = y_pred_student - y_pred_teacher
axes[1].hist(pred_diff, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero difference')
axes[1].axvline(pred_diff.mean(), color='orange', linestyle='--',
                linewidth=2, label=f'Mean diff={pred_diff.mean():.4f}')
axes[1].set_xlabel('Student - Teacher Prediction (mm)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Prediction Difference Distribution')
axes[1].legend()

plt.suptitle(
    'Knowledge Distillation -- Teacher-Student Prediction Agreement',
    fontsize=13
)
plt.tight_layout()
plt.savefig('./Visualisation/teacher_student_agreement.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: teacher_student_agreement.png")
print("\nStep 3 complete.")
print("Proceeding to Step 4 -- ONNX FP32 Export.")


# Step 4 - ONNX FP32 Export

In [16]:
# ============================================================
# STEP 4 - ONNX FP32 EXPORT
# Cell 4.1 - Convert Student MLP Pipeline to ONNX FP32
# skl2onnx converts the full sklearn Pipeline including
# StandardScaler + MLPRegressor into a single ONNX graph
# ============================================================

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

onnx_dir = './Models/onnx'
os.makedirs(onnx_dir, exist_ok=True)

fp32_path = f"{onnx_dir}/student_model_fp32.onnx"

print("=" * 60)
print("STEP 4 - ONNX FP32 EXPORT")
print("=" * 60)
print(f"Input features  : {len(feature_columns)}")
print(f"Input data type : Float32")
print(f"Output path     : {fp32_path}")
print("-" * 60)

# Define input type -- shape [None, 6] for variable batch size
initial_type = [('float_input', FloatTensorType([None, len(feature_columns)]))]

# Convert sklearn pipeline to ONNX
onnx_model_fp32 = convert_sklearn(
    student_model,
    initial_types=initial_type,
    target_opset=17
)

# Save ONNX model
with open(fp32_path, 'wb') as f:
    f.write(onnx_model_fp32.SerializeToString())

fp32_size_kb = os.path.getsize(fp32_path) / 1024

print(f"ONNX FP32 model exported successfully.")
print(f"File size : {fp32_size_kb:.2f} KB")
print(f"Saved     : {fp32_path}")


In [17]:
# ============================================================
# Cell 4.2 - Verify ONNX FP32 Predictions Match Sklearn
# Ensures the ONNX conversion did not alter model behaviour
# Acceptable tolerance: 1e-4 mm (floating point rounding only)
# ============================================================

import onnxruntime as rt

print("=" * 60)
print("ONNX FP32 VERIFICATION")
print("=" * 60)

# Create ONNX inference session
sess_fp32 = rt.InferenceSession(
    fp32_path,
    providers=['CPUExecutionProvider']
)

# Prepare test input as Float32 numpy array
X_test_fp32 = X_test.values.astype(np.float32)

# Run ONNX inference
input_name  = sess_fp32.get_inputs()[0].name
output_name = sess_fp32.get_outputs()[0].name

onnx_fp32_preds = sess_fp32.run(
    [output_name],
    {input_name: X_test_fp32}
)[0].flatten()

onnx_fp32_preds = np.clip(onnx_fp32_preds, 0, None)

# Compare with sklearn predictions
sklearn_preds = np.clip(student_model.predict(X_test), 0, None)

max_diff    = np.abs(onnx_fp32_preds - sklearn_preds).max()
mean_diff   = np.abs(onnx_fp32_preds - sklearn_preds).mean()
onnx_mae    = mean_absolute_error(y_test, onnx_fp32_preds)

print(f"Sklearn predictions sample  : {sklearn_preds[:5].round(4)}")
print(f"ONNX FP32 predictions sample: {onnx_fp32_preds[:5].round(4)}")
print(f"\nMax absolute difference  : {max_diff:.8f} mm")
print(f"Mean absolute difference : {mean_diff:.8f} mm")
print(f"ONNX FP32 MAE on test    : {onnx_mae:.4f} mm")
print("-" * 60)

TOLERANCE = 1e-4
if max_diff < TOLERANCE:
    print(f"\nVerification PASSED -- max diff {max_diff:.2e} < tolerance {TOLERANCE:.2e}")
    print("ONNX FP32 model is numerically consistent with sklearn model.")
else:
    print(f"\nVerification WARNING -- max diff {max_diff:.2e} exceeds tolerance {TOLERANCE:.2e}")
    print("Review conversion settings or opset version.")

print(f"\nONNX FP32 model ready.")
print("Proceeding to Step 5 -- INT8 Quantization.")


# Step 5 - INT8 Quantization

In [18]:
# ============================================================
# STEP 5 - INT8 QUANTIZATION
# Cell 5.1 - Dynamic Quantization FP32 --> INT8
# Dynamic quantization quantizes weights statically and
# activations dynamically at runtime -- no calibration
# dataset required, suitable for regression models
# ============================================================

from onnxruntime.quantization import quantize_dynamic, QuantType

int8_path = f"{onnx_dir}/student_model_int8.onnx"

print("=" * 60)
print("STEP 5 - INT8 QUANTIZATION")
print("=" * 60)
print(f"Input  : {fp32_path}")
print(f"Output : {int8_path}")
print(f"Method : Dynamic quantization (weights INT8, activations dynamic)")
print("-" * 60)
print("Quantizing...")

start_time = time.time()
quantize_dynamic(
    model_input=fp32_path,
    model_output=int8_path,
    weight_type=QuantType.QInt8
)
quant_time = time.time() - start_time

fp32_size_kb = os.path.getsize(fp32_path) / 1024
int8_size_kb = os.path.getsize(int8_path) / 1024
size_reduction = ((fp32_size_kb - int8_size_kb) / fp32_size_kb) * 100

print(f"\nQuantization complete in {quant_time:.2f}s")
print(f"\nModel Size Comparison:")
print(f"{'Model':<20} {'Size (KB)':>10}")
print("-" * 32)
print(f"{'FP32':<20} {fp32_size_kb:>10.2f}")
print(f"{'INT8':<20} {int8_size_kb:>10.2f}")
print(f"{'Size Reduction':<20} {size_reduction:>9.1f}%")


In [19]:
# ============================================================
# Cell 5.2 - Verify INT8 Predictions Within Tolerance
# INT8 quantization introduces larger numerical differences
# than FP32 conversion -- acceptable tolerance is 0.5 mm
# for irrigation control precision
# ============================================================

print("=" * 60)
print("INT8 QUANTIZATION VERIFICATION")
print("=" * 60)

# Create INT8 inference session
sess_int8 = rt.InferenceSession(
    int8_path,
    providers=['CPUExecutionProvider']
)

# Run INT8 inference on test set
input_name_int8  = sess_int8.get_inputs()[0].name
output_name_int8 = sess_int8.get_outputs()[0].name

onnx_int8_preds = sess_int8.run(
    [output_name_int8],
    {input_name: X_test_fp32}
)[0].flatten()

onnx_int8_preds = np.clip(onnx_int8_preds, 0, None)

# Compare INT8 vs FP32 predictions
max_diff_int8  = np.abs(onnx_int8_preds - onnx_fp32_preds).max()
mean_diff_int8 = np.abs(onnx_int8_preds - onnx_fp32_preds).mean()
int8_mae       = mean_absolute_error(y_test, onnx_int8_preds)

# Compare INT8 vs sklearn predictions
max_diff_vs_sklearn  = np.abs(onnx_int8_preds - sklearn_preds).max()
mean_diff_vs_sklearn = np.abs(onnx_int8_preds - sklearn_preds).mean()

print(f"{'Metric':<35} {'Value':>12}")
print("-" * 50)
print(f"{'INT8 vs FP32 max diff (mm)':<35} {max_diff_int8:>12.6f}")
print(f"{'INT8 vs FP32 mean diff (mm)':<35} {mean_diff_int8:>12.6f}")
print(f"{'INT8 vs Sklearn max diff (mm)':<35} {max_diff_vs_sklearn:>12.6f}")
print(f"{'INT8 vs Sklearn mean diff (mm)':<35} {mean_diff_vs_sklearn:>12.6f}")
print(f"{'INT8 MAE on test set (mm)':<35} {int8_mae:>12.4f}")
print("-" * 50)

INT8_TOLERANCE = 0.5   # 0.5 mm acceptable for irrigation control
if max_diff_int8 < INT8_TOLERANCE:
    print(f"\nVerification PASSED -- max diff {max_diff_int8:.4f} mm "
          f"< tolerance {INT8_TOLERANCE} mm")
    print("INT8 model is acceptable for edge deployment.")
else:
    print(f"\nVerification WARNING -- max diff {max_diff_int8:.4f} mm "
          f"exceeds tolerance {INT8_TOLERANCE} mm")
    print("Consider using FP32 model for deployment instead.")


In [20]:
# ============================================================
# Cell 5.3 - Full Accuracy Comparison Table
# Teacher vs Student Sklearn vs FP32 vs INT8
# Complete picture of accuracy across the pipeline
# ============================================================

all_metrics = [
    compute_metrics(y_test, y_pred_teacher,  'Hybrid Stack (Teacher)'),
    compute_metrics(y_test, sklearn_preds,   'Student MLP (Sklearn)'),
    compute_metrics(y_test, onnx_fp32_preds, 'Student MLP (ONNX FP32)'),
    compute_metrics(y_test, onnx_int8_preds, 'Student MLP (ONNX INT8)'),
]

print("=" * 70)
print("FULL ACCURACY COMPARISON")
print("=" * 70)
print(f"{'Model':<30} {'MAE (mm)':>10} {'RMSE (mm)':>10} {'R2':>8}")
print("-" * 70)
for m in all_metrics:
    print(f"{m['Model']:<30} {m['MAE']:>10.4f} {m['RMSE']:>10.4f} {m['R2']:>8.4f}")
print("-" * 70)

# Accuracy drop from teacher
print(f"\nAccuracy Drop from Teacher:")
print(f"{'Model':<30} {'MAE Drop (mm)':>14}")
print("-" * 46)
teacher_mae = all_metrics[0]['MAE']
for m in all_metrics[1:]:
    drop = m['MAE'] - teacher_mae
    sign = '+' if drop >= 0 else ''
    print(f"{m['Model']:<30} {sign}{drop:>13.4f}")
print("-" * 46)
print("\nStep 5 complete.")
print("Proceeding to Step 6 -- Benchmarking.")


# Step 6 - Benchmarking

In [21]:
# ============================================================
# STEP 6 - BENCHMARKING
# Cell 6.1 - Inference Latency Benchmark
# Measures single sample and batch inference latency
# for all four model variants
# N_WARMUP  : warmup runs to stabilise CPU cache
# N_REPEATS : timed runs for reliable average
# ============================================================

import psutil

N_WARMUP  = 10
N_REPEATS = 500

print("=" * 60)
print("STEP 6 - BENCHMARKING")
print("=" * 60)
print(f"Warmup runs  : {N_WARMUP}")
print(f"Timed runs   : {N_REPEATS}")
print(f"Machine      : {psutil.cpu_count(logical=False)} physical cores")
print(f"RAM available: {psutil.virtual_memory().available / 1024**2:.0f} MB")
print("-" * 60)

# Single sample for latency testing
X_single_sklearn = X_test.iloc[[0]]
X_single_fp32    = X_test.iloc[[0]].values.astype(np.float32)

latency_results = {}

# ---- 1. Teacher (Hybrid Stack) ----
print("Benchmarking Teacher (Hybrid Stack)...")
for _ in range(N_WARMUP):
    teacher_predict(X_single_sklearn)
times = []
for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    teacher_predict(X_single_sklearn)
    times.append((time.perf_counter() - t0) * 1000)
latency_results['Hybrid Stack (Teacher)'] = {
    'mean_ms': np.mean(times),
    'std_ms':  np.std(times),
    'p95_ms':  np.percentile(times, 95)
}

# ---- 2. Student MLP Sklearn ----
print("Benchmarking Student MLP (Sklearn)...")
for _ in range(N_WARMUP):
    student_model.predict(X_single_sklearn)
times = []
for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    student_model.predict(X_single_sklearn)
    times.append((time.perf_counter() - t0) * 1000)
latency_results['Student MLP (Sklearn)'] = {
    'mean_ms': np.mean(times),
    'std_ms':  np.std(times),
    'p95_ms':  np.percentile(times, 95)
}

# ---- 3. Student MLP ONNX FP32 ----
print("Benchmarking Student MLP (ONNX FP32)...")
for _ in range(N_WARMUP):
    sess_fp32.run([output_name], {input_name: X_single_fp32})
times = []
for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    sess_fp32.run([output_name], {input_name: X_single_fp32})
    times.append((time.perf_counter() - t0) * 1000)
latency_results['Student MLP (ONNX FP32)'] = {
    'mean_ms': np.mean(times),
    'std_ms':  np.std(times),
    'p95_ms':  np.percentile(times, 95)
}

# ---- 4. Student MLP ONNX INT8 ----
print("Benchmarking Student MLP (ONNX INT8)...")
for _ in range(N_WARMUP):
    sess_int8.run([output_name_int8], {input_name_int8: X_single_fp32})
times = []
for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    sess_int8.run([output_name_int8], {input_name_int8: X_single_fp32})
    times.append((time.perf_counter() - t0) * 1000)
latency_results['Student MLP (ONNX INT8)'] = {
    'mean_ms': np.mean(times),
    'std_ms':  np.std(times),
    'p95_ms':  np.percentile(times, 95)
}

print("\nLatency benchmark complete.")
print(f"\n{'Model':<30} {'Mean (ms)':>10} {'Std (ms)':>10} {'P95 (ms)':>10}")
print("-" * 65)
for model_name, metrics in latency_results.items():
    print(f"{model_name:<30} {metrics['mean_ms']:>10.4f} "
          f"{metrics['std_ms']:>10.4f} {metrics['p95_ms']:>10.4f}")


In [22]:
# ============================================================
# Cell 6.2 - RAM Usage Benchmark
# Measures memory footprint during inference for each model
# Uses psutil to track process memory before and during
# inference to isolate model-specific RAM consumption
# ============================================================

import gc

def measure_ram_usage(predict_fn, n_runs=50):
    """
    Measures peak RAM usage during inference.
    Returns RAM used in KB.
    """
    gc.collect()
    process   = psutil.Process(os.getpid())
    ram_before = process.memory_info().rss / 1024   # KB

    for _ in range(n_runs):
        predict_fn()

    ram_after = process.memory_info().rss / 1024    # KB
    return max(ram_after - ram_before, 0)

print("=" * 60)
print("RAM USAGE BENCHMARK")
print("=" * 60)

ram_results = {}

# Teacher
ram_results['Hybrid Stack (Teacher)'] = measure_ram_usage(
    lambda: teacher_predict(X_single_sklearn)
)

# Student Sklearn
ram_results['Student MLP (Sklearn)'] = measure_ram_usage(
    lambda: student_model.predict(X_single_sklearn)
)

# Student ONNX FP32
ram_results['Student MLP (ONNX FP32)'] = measure_ram_usage(
    lambda: sess_fp32.run([output_name], {input_name: X_single_fp32})
)

# Student ONNX INT8
ram_results['Student MLP (ONNX INT8)'] = measure_ram_usage(
    lambda: sess_int8.run([output_name_int8], {input_name_int8: X_single_fp32})
)

print(f"{'Model':<30} {'RAM Usage (KB)':>15}")
print("-" * 48)
for model_name, ram_kb in ram_results.items():
    print(f"{model_name:<30} {ram_kb:>15.2f}")

print("\nNote: RAM measurements are approximate.")
print("      Pi measurements will differ due to hardware differences.")


In [25]:
# ============================================================
# Cell 6.3 - Full Deployment Metrics Table
# Consolidates accuracy, latency, RAM and file size
# into one deployment readiness summary
# ============================================================

# Save sklearn student model first
sklearn_pkl_path = f'{onnx_dir}/student_model_sklearn.pkl'
joblib.dump(student_model, sklearn_pkl_path)

# File sizes
sklearn_size_kb = os.path.getsize(sklearn_pkl_path) / 1024
fp32_size_kb    = os.path.getsize(fp32_path) / 1024
int8_size_kb    = os.path.getsize(int8_path) / 1024

# Calculate total teacher size across all 4 component files
teacher_files = [
    f'./Models/retrained/retrained_family1_catboost.pkl',
    f'./Models/retrained/retrained_family2_svr.pkl',
    f'./Models/retrained/retrained_family3_mlp.pkl',
    f'./Models/meta_learner_ridge.pkl'
]
teacher_size_kb = sum(os.path.getsize(f) / 1024 for f in teacher_files)

print("=" * 90)
print("FULL DEPLOYMENT METRICS TABLE (Development Machine)")
print("=" * 90)
print(f"{'Model':<30} {'MAE (mm)':>9} {'Size (KB)':>10} "
      f"{'Latency Mean (ms)':>18} {'P95 (ms)':>10} {'RAM (KB)':>10}")
print("-" * 90)

deployment_data = [
    {
        'name':    'Hybrid Stack (Teacher)',
        'mae':     all_metrics[0]['MAE'],
        'size_kb': teacher_size_kb,
        'latency': latency_results['Hybrid Stack (Teacher)'],
        'ram_kb':  ram_results['Hybrid Stack (Teacher)']
    },
    {
        'name':    'Student MLP (Sklearn)',
        'mae':     all_metrics[1]['MAE'],
        'size_kb': sklearn_size_kb,
        'latency': latency_results['Student MLP (Sklearn)'],
        'ram_kb':  ram_results['Student MLP (Sklearn)']
    },
    {
        'name':    'Student MLP (ONNX FP32)',
        'mae':     all_metrics[2]['MAE'],
        'size_kb': fp32_size_kb,
        'latency': latency_results['Student MLP (ONNX FP32)'],
        'ram_kb':  ram_results['Student MLP (ONNX FP32)']
    },
    {
        'name':    'Student MLP (ONNX INT8)',
        'mae':     all_metrics[3]['MAE'],
        'size_kb': int8_size_kb,
        'latency': latency_results['Student MLP (ONNX INT8)'],
        'ram_kb':  ram_results['Student MLP (ONNX INT8)']
    },
]

for d in deployment_data:
    size_str = f"{d['size_kb']:.2f}" if d['size_kb'] is not None else 'N/A'
    print(f"{d['name']:<30} {d['mae']:>9.4f} {size_str:>10} "
          f"{d['latency']['mean_ms']:>18.4f} "
          f"{d['latency']['p95_ms']:>10.4f} "
          f"{d['ram_kb']:>10.2f}")

print("-" * 90)
print("\nNote: These are development machine benchmarks.")
print("      Raspberry Pi 4 benchmarks will be collected in the deployment phase.")
print("\nStep 6 complete.")
print("Proceeding to Step 7 -- Save All Phase 3 Artefacts.")


# Step 7 - Saving all Phase 3 artefacts

In [26]:
# ============================================================
# STEP 7 - SAVE ALL PHASE 3 ARTEFACTS
# Cell 7.1 - Save Student Model and ONNX Files
# Fixed filenames -- reruns overwrite existing files
# ============================================================

phase3_dir = './Models/phase3'
os.makedirs(phase3_dir, exist_ok=True)

print("=" * 60)
print("STEP 7 - SAVE ALL PHASE 3 ARTEFACTS")
print("=" * 60)
print(f"Save directory : {phase3_dir}")
print("-" * 60)

# --- Save Student Sklearn Pipeline ---
student_sklearn_path = f"{phase3_dir}/student_model_sklearn.pkl"
joblib.dump(student_model, student_sklearn_path)
print(f"Student sklearn saved  : {student_sklearn_path} "
      f"({os.path.getsize(student_sklearn_path)/1024:.2f} KB)")

# --- Copy ONNX FP32 ---
import shutil
student_fp32_path = f"{phase3_dir}/student_model_fp32.onnx"
student_int8_path = f"{phase3_dir}/student_model_int8.onnx"

shutil.copy2(fp32_path, student_fp32_path)
shutil.copy2(int8_path, student_int8_path)

print(f"Student FP32 saved     : {student_fp32_path} "
      f"({os.path.getsize(student_fp32_path)/1024:.2f} KB)")
print(f"Student INT8 saved     : {student_int8_path} "
      f"({os.path.getsize(student_int8_path)/1024:.2f} KB)")

print("\nAll model files saved successfully.")


In [27]:
# ============================================================
# Cell 7.2 - Save Distillation Data
# Saves soft labels and blended targets for reproducibility
# ============================================================

distill_dir = f"{phase3_dir}/distillation_data"
os.makedirs(distill_dir, exist_ok=True)

# Save distillation arrays
np.save(f"{distill_dir}/X_distill.npy",          X_distill.values)
np.save(f"{distill_dir}/y_distill_ground_truth.npy", y_distill.values)
np.save(f"{distill_dir}/y_teacher_soft_labels.npy",  y_teacher)
np.save(f"{distill_dir}/y_distill_blended.npy",      y_distill_blended)

print("=" * 60)
print("DISTILLATION DATA SAVED")
print("=" * 60)
print(f"X_distill              : {X_distill.shape}")
print(f"y_distill_ground_truth : {y_distill.shape}")
print(f"y_teacher_soft_labels  : {y_teacher.shape}")
print(f"y_distill_blended      : {y_distill_blended.shape}")
print(f"\nSaved to : {distill_dir}")


In [29]:
# ============================================================
# Cell 7.4 - Verify All Saved Phase 3 Files
# ============================================================

print("=" * 60)
print("STEP 7.4 - SAVED FILE VERIFICATION")
print("=" * 60)

files_to_verify = [
    student_sklearn_path,
    student_fp32_path,
    student_int8_path,
    f"{distill_dir}/X_distill.npy",
    f"{distill_dir}/y_distill_ground_truth.npy",
    f"{distill_dir}/y_teacher_soft_labels.npy",
    f"{distill_dir}/y_distill_blended.npy"
]

all_verified = True

for filepath in files_to_verify:
    exists  = os.path.exists(filepath)
    size_kb = os.path.getsize(filepath) / 1024 if exists else 0
    status  = 'OK' if exists else 'MISSING'
    if not exists:
        all_verified = False
    print(f"{status:<8} {size_kb:>8.2f} KB   {filepath}")

print("-" * 60)
if all_verified:
    print("\nAll Phase 3 artefacts verified successfully.")
    print("\nPhase 3 complete. Ready for Raspberry Pi deployment.")
    print("\nFiles to copy to Raspberry Pi:")
    print(f"  --> {student_fp32_path}")
    print(f"  --> {student_int8_path}")
else:
    print("\nWarning: Some files are missing. Re-run the affected cells.")
